In [ ]:
import math
import pandas as pd
import re


In [ ]:
df = pd.read_csv('data/movies.csv')
df.head()

,Unnamed: 0,id,title,overview,release_date,popularity,vote_average,vote_count
0,0,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...",1995-10-20,18.433,8.7,2763
1,1,724089,Gabriel's Inferno Part II,Professor Gabriel Emerson finally learns the t...,2020-07-31,8.439,8.7,1223
2,2,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,1994-09-23,65.570,8.7,18637
3,3,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",1972-03-14,63.277,8.7,14052
4,4,761053,Gabriel's Inferno Part III,The final part of the film adaption of the ero...,2020-11-19,26.691,8.7,773


In [3]:
df = df[['title','overview']]
df.shape

(8551, 2)

In [4]:
print(df.isna().sum())
#drop rows with null value
df = df.dropna()
df

title        0
overview    20
dtype: int64


,title,overview
0,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second..."
1,Gabriel's Inferno Part II,Professor Gabriel Emerson finally learns the t...
2,The Shawshank Redemption,Framed in the 1940s for the double murder of h...
3,The Godfather,"Spanning the years 1945 to 1955, a chronicle o..."
4,Gabriel's Inferno Part III,The final part of the film adaption of the ero...
...,...,...
8546,Disaster Movie,"The filmmaking team behind the hits ""Scary Mov..."
8547,Battlefield Earth,"In the year 3000, man is no match for the Psyc..."
8548,Dragonball Evolution,The young warrior Son Goku sets out on a quest...
8549,House of the Dead,"Set on an island off the coast, a techno rave ..."


In [5]:
#resample the df to 500
df_resample = df.sample(500,random_state=42,replace=False).reset_index(drop=True)
print('Shape:' ,df_resample.shape)
df_resample['overview'] = df_resample['overview'].str.lower()
df_resample.head()

Shape: (500, 2)


,title,overview
0,Escape from New York,"in 1997, the island of manhattan has been wall..."
1,Casper: A Spirited Beginning,a prequel to the first animated/live-action ca...
2,Into the Blue,when they take some friends on an extreme spor...
3,Brahms: The Boy II,after a family moves into the heelshire mansio...
4,The Dark Half,thad beaumont is the author of a highly succes...


In [10]:
def preprocess_text(text):
    """
    Lowercase, remove non-alphabetic characters, and split on whitespace.
    Returns a list of tokens.
    """
    text = text.lower()                        # lowercase
    text = re.sub(r'[^a-z\s]', '', text)       # remove punctuation/numbers
    tokens = text.split()                      # split on whitespace
    return tokens

def build_vocabulary(documents):
    """
    documents: list of strings (movie overviews)
    Returns a sorted list of unique tokens (the 'vocabulary').
    """
    vocab_set = set()
    for doc in documents:
        tokens = preprocess_text(doc)
        vocab_set.update(tokens)
    vocabulary = sorted(list(vocab_set))
    return vocabulary

def create_token_index_map(vocabulary):
    """
    Create a dictionary mapping each token -> index in the vocabulary list.
    This makes it faster to do token lookups later.
    """
    return {token: i for i, token in enumerate(vocabulary)}

def compute_idf(documents, vocabulary, token2idx):
    """
    documents: list of strings (movie overviews)
    vocabulary: global list of tokens
    token2idx: dict mapping token -> index in vocabulary
    Returns a list of IDF values, aligned with 'vocabulary'.
    """
    N = len(documents)  # number of documents
    doc_count = [0] * len(vocabulary)  # doc_count[i] = number of docs containing vocabulary[i]

    for doc in documents:
        tokens_in_doc = set(preprocess_text(doc))  # unique tokens in this doc
        for token in tokens_in_doc:
            if token in token2idx:
                idx = token2idx[token]
                doc_count[idx] += 1

    idf_values = []
    for i in range(len(vocabulary)):
        # IDF formula: log(N / (1 + count))
        # +1 in the denominator to avoid division by zero
        idf = math.log(N / (1 + doc_count[i]))
        idf_values.append(idf)

    return idf_values

def compute_tf_vector(tokens, token2idx):
    """
    tokens: list of tokens from ONE document (or query)
    token2idx: dictionary {token -> index}
    Returns a list (vector) of term frequencies (raw count).
    """
    tf_vector = [0] * len(token2idx)
    for token in tokens:
        if token in token2idx:
            idx = token2idx[token]
            tf_vector[idx] += 1
    return tf_vector

def compute_tfidf_vector(doc, token2idx, idf_values):
    """
    doc: string (overview or user query)
    token2idx: dictionary {token -> index}
    idf_values: list of IDF values for each token in vocabulary
    Returns a TF-IDF vector (list of floats).
    """
    tokens = preprocess_text(doc)
    tf_vector = compute_tf_vector(tokens, token2idx)
    tfidf_vector = [tf_vector[i] * idf_values[i] for i in range(len(tf_vector))]
    return tfidf_vector


def cosine_similarity(vec_a, vec_b):
    """
    Returns the cosine similarity between two vectors
    """
    # dot product
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    # norms
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))

    #make sure the denominator is not 0
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)





In [ ]:
documents = df_resample['overview'].astype(str).tolist()
# Build vocabulary and supporting structures
vocabulary = build_vocabulary(documents)
token2idx = create_token_index_map(vocabulary)

# Compute IDF array
idf_values = compute_idf(documents, vocabulary, token2idx)

# Pre-compute each movie's TF-IDF vector
# I store them in a list of tuples: (movie_index, title, tfidf_vector)
doc_vectors = []
for i, overview in enumerate(documents):
    vec = compute_tfidf_vector(overview, token2idx, idf_values)
    title = df.iloc[i]['title']
    doc_vectors.append((i, title, vec))

user_input = input ('Enter the movie you like to see')


#Compute TF-IDF vector for the user query
user_vector = compute_tfidf_vector(user_input, token2idx, idf_values)

# Compute similarity to each document
similarities = []
for (i, title, doc_vec) in doc_vectors:
    sim = cosine_similarity(user_vector, doc_vec)
    similarities.append((i, title, sim))

# Sort by similarity, descending and pick the top N
similarities.sort(key=lambda x: x[2], reverse=True)
top_results = similarities[:5]


print(f"\nUser Input: '{user_input}'\n")
print("Recommended Movies:")
for idx, title, score in top_results:
    print(f"{title} Similarity: {score:.4f}")
    print(df_resample.iloc[idx]['overview'])
    print()



User Input: 'I like action movies set in space'

Recommended Movies:
Captain Fantastic (Similarity: 0.2219)
space pirate captain harlock and his fearless crew face off against the space invaders who seek to conquer the planet earth.

A Clockwork Orange (Similarity: 0.1385)
4 would be astronauts spend 400 days in a land locked space simulator to test the psychological effects of deep space travel but, when something goes terribly wrong and they are forced to leave the simulation, they discover that everything on earth has changed. is this real or is the simulation on a higher level than they could have ever imagined?

Mulan (Similarity: 0.1306)
a faithful wife takes action when it becomes clear to her that her devious husband has betrayed her.

The Night of the Hunter (Similarity: 0.1254)
a chronicle of the original mercury astronauts in the formation of america's space program: alan shepherd, the first american in space; gus grissom, the benighted astronaut for whom nothing works out 